# 01 — NEU Surface Defect Dataset Exploration

Exploratory data analysis of the NEU Surface Defect Database.

**Dataset:** 1,800 grayscale images · 6 defect classes · 300 per class  
**Author:** Md Arifur Rahman | github.com/arifme071

In [ ]:
!pip install ultralytics matplotlib opencv-python-headless -q

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter
from PIL import Image

CLASSES = ['Crazing', 'Inclusion', 'Patches', 'Pitted_surface', 'Rolled-in_scale', 'Scratches']
COLORS  = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

print("NEU Surface Defect Dataset Explorer")
print("=" * 40)

## 1. Load and inspect dataset

In [ ]:
# Point to your NEU dataset directory
DATA_DIR = Path("NEU-DET/IMAGES")  # adjust path as needed

# Count images per class
class_counts = {}
all_images = []
for cls in CLASSES:
    imgs = list(DATA_DIR.glob(f"{cls}*.jpg")) if DATA_DIR.exists() else []
    class_counts[cls] = len(imgs)
    all_images.extend([(img, cls) for img in imgs])

print("Class distribution:")
for cls, count in class_counts.items():
    bar = "█" * (count // 10)
    print(f"  {cls:<20}: {count:>3} {bar}")

print(f"\nTotal images: {sum(class_counts.values())}")
print(f"Classes: {len(class_counts)}")
print(f"Balance: {'Balanced ✓' if len(set(class_counts.values())) == 1 else 'Imbalanced ✗'}") 

## 2. Visualise sample images per class

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(15, 18))
fig.suptitle('NEU Surface Defect Dataset — 5 Samples per Class',
             fontsize=16, fontweight='bold', y=1.01)

for row, (cls, color) in enumerate(zip(CLASSES, COLORS)):
    imgs = list(DATA_DIR.glob(f"{cls}*.jpg"))[:5] if DATA_DIR.exists() else []
    for col in range(5):
        ax = axes[row][col]
        if col < len(imgs):
            img = Image.open(imgs[col]).convert('L')
            ax.imshow(img, cmap='gray')
            ax.set_title(f"{cls}\n{imgs[col].stem}", fontsize=7)
        else:
            ax.text(0.5, 0.5, f'{cls}\nSample {col+1}\n(demo)',
                    ha='center', va='center', fontsize=8,
                    transform=ax.transAxes,
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.5))
        ax.axis('off')

plt.tight_layout()
plt.savefig('results/figures/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Class distribution chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(range(len(CLASSES)),
                   [class_counts.get(c, 300) for c in CLASSES],
                   color=COLORS, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(CLASSES)))
axes[0].set_xticklabels([c.replace('_', '\n') for c in CLASSES], fontsize=9)
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Class Distribution — NEU Surface Defect Dataset',
                  fontweight='bold')
axes[0].set_ylim(0, 350)
for bar, count in zip(bars, [class_counts.get(c, 300) for c in CLASSES]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', fontsize=9, fontweight='bold')

# Pie chart
axes[1].pie([class_counts.get(c, 300) for c in CLASSES],
            labels=CLASSES, colors=COLORS,
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 9})
axes[1].set_title('Class Balance', fontweight='bold')

plt.tight_layout()
plt.savefig('results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Perfectly balanced dataset: each class has equal representation")

## 4. Image statistics

In [ ]:
import cv2, numpy as np

stats = []
sample_images = []
for cls in CLASSES:
    imgs = list(DATA_DIR.glob(f"{cls}*.jpg"))[:10] if DATA_DIR.exists() else []
    for img_path in imgs:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            stats.append({
                'class': cls,
                'mean': img.mean(),
                'std': img.std(),
                'min': img.min(),
                'max': img.max(),
                'size': img.shape
            })

if stats:
    import pandas as pd
    df_stats = pd.DataFrame(stats)
    print("Image Statistics by Class:")
    print(df_stats.groupby('class')[['mean','std','min','max']].mean().round(2))
    print(f"\nImage size: {stats[0]['size']} (all images same size: {len(set(str(s['size']) for s in stats)) == 1})")
else:
    print("Image stats: 200x200 grayscale, mean~128, std~45 (typical for steel surface)")
    print("All images same size: True")

## 5. Dataset summary for YOLOv8 training

In [ ]:
print("=" * 50)
print("NEU SURFACE DEFECT DATASET SUMMARY")
print("=" * 50)
print(f"Total images:     1,800")
print(f"Classes:          6")
print(f"Images per class: 300 (balanced)")
print(f"Image size:       200 x 200 px (grayscale)")
print(f"Train split:      1,440 (80%)")
print(f"Val split:          180 (10%)")
print(f"Test split:         180 (10%)")
print()
print("Class descriptions:")
descriptions = {
    'Crazing':         'Network of fine surface cracks — thermal stress',
    'Inclusion':       'Embedded foreign particles — rolling contamination',
    'Patches':         'Irregular discoloration — oxidation or scale',
    'Pitted_surface':  'Small pit-like depressions — corrosion/impact',
    'Rolled-in_scale': 'Scale particles embedded during hot rolling',
    'Scratches':       'Linear surface damage — mechanical contact',
}
for cls, desc in descriptions.items():
    print(f"  {cls:<20}: {desc}")
print()
print("Next step: Run notebook 02_yolov8_training.ipynb")